### Section A — OOP & Encapsulation 

Q1. User Profile with Encapsulation 

Topics: Encapsulation, Validation 

Problem Statement: 

Create a User class with private attributes: _username, _email, and _age. Provide getter and setter methods. The setter for email must validate that it contains '@' and '.' characters. The setter for age must ensure value is between 18 and 120. 

Input: 

user = User("alice", "alice@mail.com", 25) 

user.set_email("invalid") 

user.set_age(150) 

print(user.get_email()) 

print(user.get_age()) 

Output: 

ValueError: Invalid email format 

ValueError: Age must be between 18 and 120 

alice@mail.com 

25 

Constraints: 

Use private variables (prefix with _) 

Raise ValueError with descriptive messages on invalid input 

No direct attribute access allowed from outside the class 

In [16]:
class User:
    def __init__(self,username,email,age):
        self._username=username
        self.set_email(email)
        self.set_age(age)
    def get_email(self):
        return self._email
    def set_email(self,email):
        if '@' not in email or '.' not in email:
            raise ValueError("invalid email format")
        self._email=email

    def get_age(self):
        return self._age
    def set_age(self,age):
        if age < 18 or age > 120:
            raise ValueError("age must be between 18 to 120")
        self._age=age
    def get_username(self):
        return self._username
    
user = User("alice", "alice@gmail.com", 25) 

try:
    user.set_email("sushmagmail.com")
except ValueError as e:
    print(e)
try:
    user.set_age(150)
except ValueError as e:
    print(e)

print(user.get_email()) 

print(user.get_age()) 

invalid email format
age must be between 18 to 120
alice@gmail.com
25


### Q2. Inheritance — Admin and Customer Users 

Topics: Inheritance, Polymorphism, super() 

Problem Statement: 

Create a base class User with attributes username and role, and a method display_profile(). Create two subclasses: AdminUser (extra attribute: permissions list) and CustomerUser (extra attribute: orders count). Override display_profile() in both subclasses to include their specific data. 

Input: 

admin = AdminUser("admin1", ["manage_users", "view_logs"]) 

customer = CustomerUser("cust1", 5) 

admin.display_profile() 

customer.display_profile() 

Output: 

Admin: admin1 | Permissions: manage_users, view_logs 

Customer: cust1 | Orders: 5 

Constraints: 

Use super().__init__() to initialize base class 

Override display_profile() in each subclass (Polymorphism) 

Permissions must be a list of strings 

 
 

In [22]:
class User:

    def __init__(self,username,role):
        self.username=username
        self.role=role
    
    def display_profile(self):
        return f"{self.username} and {self.role}"
    
class AdminUser(User):
    def __init__(self,username,permissions):
        super().__init__(username,"Admin")
        self.permissions=permissions
    def display_profile(self):
        perms=','.join(self.permissions)
        print(f"admin :{self.username} | permissions : {perms}")

class Customer(User):
    def __init__(self,username,orders):
        super().__init__(username,"customer")
        self.orders=orders

    def display_profile(self):
       
        print(f"customer: {self.username} | orders :{self.orders}")
admin=AdminUser("admin1",["manage_users","view_logs"])
customer=Customer("cust1",4)
admin.display_profile()
customer.display_profile()

admin :admin1 | permissions : manage_users,view_logs
customer: cust1 | orders :4


### Q3. Composition — Order with Address and Payment 

Topics: Composition, SRP 

Problem Statement: 

Build an Order class using composition. Create separate classes: Address (city, zip_code), PaymentInfo (method, amount), and OrderItem (name, qty, price). The Order class should contain an Address, PaymentInfo, and a list of OrderItems. Implement order_summary() that returns the full breakdown. 

Input: 

addr = Address("Bangalore", "560001") 

pay = PaymentInfo("UPI", 1500) 

items = [OrderItem("Book", 2, 500), OrderItem("Pen", 5, 100)] 

order = Order(addr, pay, items) 

order.order_summary() 

Output: 

Shipping: Bangalore - 560001 

Items: Book x2 = 1000, Pen x5 = 500 

Total: 1500 

Payment: UPI 

Constraints: 

Do NOT use inheritance; use composition only 

Each class must have a single responsibility 

OrderItem total = qty * price 

In [24]:
class Address:
    def __init__(self, city, zip_code):
        self.city = city
        self.zip_code = zip_code


class PaymentInfo:
    def __init__(self, method, amount):
        self.method = method
        self.amount = amount


class OrderItem:
    def __init__(self, name, qty, price):
        self.name = name
        self.qty = qty
        self.price = price

    def total(self):
        return self.qty * self.price   # ✅ qty * price


class Order:
    def __init__(self, address, payment, items):
        self.address = address            # Composition
        self.payment = payment            # Composition
        self.items = items                # List of OrderItem objects

    def order_summary(self):
        # Shipping info
        print(f"Shipping: {self.address.city} - {self.address.zip_code}")

        # Items breakdown
        item_strings = []
        total = 0
        for item in self.items:
            item_total = item.total()
            total += item_total
            item_strings.append(f"{item.name} x{item.qty} = {item_total}")

        print("Items:", ", ".join(item_strings))

        # Total and payment
        print(f"Total: {total}")
        print(f"Payment: {self.payment.method}")
        
addr = Address("Bangalore", "560001")
pay = PaymentInfo("UPI", 1500)

items = [
    OrderItem("Book", 2, 500),
    OrderItem("Pen", 5, 100)
]

order = Order(addr, pay, items)
order.order_summary()

Shipping: Bangalore - 560001
Items: Book x2 = 1000, Pen x5 = 500
Total: 1500
Payment: UPI


### Section B — SOLID Principles 

Q4. SRP — Separate Validation, Storage, and Notification 

Topics: SRP 

Problem Statement: 

You are given a monolithic UserService class that validates user data, saves to a JSON file, and sends a welcome message — all in one method. Refactor it into three SRP-compliant classes: UserValidator, UserStorage, and UserNotifier. Write an orchestrator function that uses all three. 

Input: 

data = {"username": "alice", "email": "alice@mail.com"} 

register_user(data) 

Output: 

Validation passed 

User saved to users.json 

Welcome email sent to alice@mail.com 

Constraints: 

Each class must have exactly one reason to change 

UserValidator raises ValueError on invalid data 

UserStorage reads/writes JSON 

UserNotifier prints the notification (simulated) 

In [29]:
import json 
import os

class UserValidator:
    def validate(self,data):
        if "username" not in data or not data["username"]:
            raise ValueError("username required")

        if "email" not in data or '@' not in data["email"] or '.' not in data['email']:
            raise ValueError("invalid email format")
        print("Validation passed")
class UserStorage:
    def __init__(self,filename="users.json"):
        self.filename=filename
    def save(self,data):
        users=[]

        if os.path.exists(self.filename):
            with open(self.filename,'r')as f:
                try:
                    users=json.load(f)
                except json.JSONDecodeError:
                    users=[]

        users.append(data)

        with open(self.filename,'w')as f:
            json.dump(users,f,indent=4)
        print("user saved to users.json")

class UserNotifier:
    def send_welcome(self,email):
        print(f"welcome email sent to {email}")

# Orchestrator function
def register_user(data):
    validator = UserValidator()
    storage = UserStorage()
    notifier = UserNotifier()

    # Step 1: Validate
    validator.validate(data)

    # Step 2: Save
    storage.save(data)

    # Step 3: Notify
    notifier.send_welcome(data["email"])

data = {"username": "alice", "email": "alice@mail.com"}

register_user(data)

Validation passed
user saved to users.json
welcome email sent to alice@mail.com


### Q5. OCP — Extensible Discount System 

Topics: OCP, Abstraction 

Problem Statement: 

Create a discount system where new discount types can be added without modifying existing code. Define an abstract base class Discount with method apply(amount). Implement: NoDiscount, PercentageDiscount (10%), FlatDiscount (Rs 200 off), and BuyOneGetOneFree (50% off). Write a function calculate_total(amount, discount). 

Input: 

print(calculate_total(1000, PercentageDiscount())) 

print(calculate_total(1000, FlatDiscount())) 

print(calculate_total(1000, BuyOneGetOneFree())) 

Output: 

900.0 

800.0 

500.0 

Constraints: 

Use ABC and @abstractmethod 

Adding a new discount type must NOT modify calculate_total() 

Discount cannot return negative amounts; minimum is 0 

 

In [ ]:
from abc import ABC, abstractmethod


# Abstract Base Class
class Discount(ABC):
    @abstractmethod
    def apply(self, amount):
        pass
# No Discount
class NoDiscount(Discount):
    def apply(self, amount):
        return amount


# Percentage Discount (10%)
class PercentageDiscount(Discount):
    def apply(self, amount):
        return max(0, amount * 0.9)


# Flat Discount (₹200 off)
class FlatDiscount(Discount):
    def apply(self, amount):
        return max(0, amount - 200)


# Buy One Get One Free (50% off)
class BuyOneGetOneFree(Discount):
    def apply(self, amount):
        return max(0, amount * 0.5)


# Function that remains unchanged (OCP )
def calculate_total(amount, discount: Discount):
    return discount.apply(amount)


print(calculate_total(1000, PercentageDiscount()))
print(calculate_total(1000, FlatDiscount()))
print(calculate_total(1000, BuyOneGetOneFree()))

900.0
800
500.0


### 6. LSP — Fix the Bird Hierarchy 

Topics: LSP, ISP, Abstraction 

Problem Statement: 

The following design violates LSP: 

class Bird: def fly(self): ... 

class Penguin(Bird): def fly(self): raise Exception("Can't fly") 

Redesign the hierarchy so that no subclass breaks the contract of its parent. Create proper abstract classes: Bird, FlyingBird, SwimmingBird. Implement Sparrow, Eagle, Penguin, and Duck (Duck both flies and swims). 

Input: 

for bird in [Sparrow(), Eagle(), Penguin(), Duck()]: 

    bird.move() 

Output: 

Sparrow flies 

Eagle flies 

Penguin swims 

Duck flies and swims 

Constraints: 

No method should raise an unexpected exception 

Use abstract base classes with ABC 

Duck must implement both flying and swimming interfaces 

LSP: every subclass must be substitutable for its parent 

In [ ]:
from abc import ABC, abstractmethod

class Bird(ABC):
    """ Bird is the base class = all birds share a name/identity, but not movement"""
    @abstractmethod
    def move(self):
        pass
class FlyingBird(Bird):
    def fly(self):
        return f"{self.__class__.__name__} flies"
    def move(self):
        print(self.fly())
class SwimmingBird(Bird):
    def swim(self):
        return f"{self.__class__.__name__} swims"
    def move(self):
        print(self.swim())
class Sparrow(FlyingBird):
    print("yes it fly")
class Eagle(FlyingBird):
    pass
class Penguin(SwimmingBird):
    print('yes it swims')
class Duck(FlyingBird,SwimmingBird):
    """ Duck can both fly and swim-overrides move() to do both."""
    def move(self):
        print(f"{self.__class__.__name__} flies and swims")
for bird in [Sparrow(),Eagle(),Penguin(),Duck()]:
    bird.move()



yes it fly
yes it swims
Sparrow flies
Eagle flies
Penguin swims
Duck flies and swims


### Q7. DIP — Repository Pattern with Dependency Injection 

Topics: DIP, Dependency Injection 

Problem Statement: 

Create a UserRepository abstract class with methods save(user) and find(username). Implement JSONUserRepository (stores in JSON file) and InMemoryUserRepository (stores in a dict). Create a UserService that depends on the abstraction, not the concrete class. Inject the repository at runtime. 

Input: 

repo = InMemoryUserRepository() 

service = UserService(repo) 

service.register({"username": "alice", "email": "a@b.com"}) 

print(service.get_user("alice")) 

Output: 

{'username': 'alice', 'email': 'a@b.com'} 

Constraints: 

UserService constructor must accept UserRepository (abstraction) 

Swapping InMemoryUserRepository with JSONUserRepository must not change UserService code 

Use ABC for the repository interface 

In [4]:
from abc import ABC, abstractmethod
import json,os

#abstarction(interface)
class UserRepository(ABC):
    @abstractmethod
    def save(self,user:dict):pass

    @abstractmethod
    def find(self,username:str)->dict:pass

class InMemoryUserRepository(UserRepository):
    def __init__(self):
        self._store={}
    def save(self,user):
        self._store[user["username"]]=user
    def find(self,username):
        return self._store.get(username)
class JSONUserRepositoy(UserRepository):
    def __init__(self,filepath="users.json"):
        self._path=filepath
        if not os.path.exists(filepath):
            with open(filepath,"w")as f:
                json.dump({},f)
    def _load(self):
        with open(self._path)as f:
            data= json.load(f)
            if isinstance(data,list):
                return {}
            return data
    def _dump(self,data):
        with open(self._path,"w")as f:
            json.dump(data,f,indent=2)

    def save(self,user):
        data=self._load()
        data[user["username"]]=user
        self._dump(data)
    def find(self,username):
        return self._load().get(username)

#service depends on abstarction, not concrete class
class UserService:
    def __init__(self,repo:UserRepository):
        self._repo=repo
    def register(self,user:dict):
        self._repo.save(user)
    def get_user(self,username:str):
        return self._repo.find(username)
    
#usage is swap the repo without changing UserService
repo=InMemoryUserRepository()
service=UserService(repo)
service.register({"username":"sushma","email":"sangolli@gmail.com"})
print(service.get_user("sushma"))

# Swap to JSON — UserService code is untouched
json_repo = JSONUserRepositoy()
service2 = UserService(json_repo)
service2.register({"username": "airhant", "email": "arihant@gmail.com"})
print(service2.get_user("bob"))
# {'username': 'bob', 'email': 'b@b.com'}

{'username': 'sushma', 'email': 'sangolli@gmail.com'}
{'username': 'bob', 'email': 'b@b.com'}


### Section C — Threading, Multiprocessing & Async 

Q8. Thread vs Sequential — IO Simulation 

Topics: Threading, IO-bound 

Problem Statement: 

Create a function fetch_data(source, delay) that simulates an API call using time.sleep(delay). Run 5 calls sequentially and then using threading. Print the total execution time for each approach. 

Input: 

sources = [("users", 2), ("orders", 3), ("products", 1), ("reviews", 2), ("inventory", 1)] 

Output: 

Sequential time: ~9.0s 

Threaded time:   ~3.0s 

Constraints: 

Use threading.Thread or concurrent.futures.ThreadPoolExecutor 

Print start and end log for each source 

Measure time using time.time() 

 

In [41]:
from abc import ABC, abstractmethod
import json, os

# --- Abstraction (interface) ---
class UserRepository(ABC):
    @abstractmethod
    def save(self, user: dict): pass

    @abstractmethod
    def find(self, username: str) -> dict: pass


# --- Concrete: in-memory ---
class InMemoryUserRepository(UserRepository):
    def __init__(self):
        self._store = {}

    def save(self, user):
        self._store[user["username"]] = user

    def find(self, username):
        return self._store.get(username)


# --- Concrete: JSON file ---
class JSONUserRepository(UserRepository):
    def __init__(self, filepath="users.json"):
        self._path = filepath
        if not os.path.exists(filepath):
            with open(filepath, "w") as f:
                json.dump({}, f)

    def _load(self):
        with open(self._path) as f:
            return json.load(f)

    def _dump(self, data):
        with open(self._path, "w") as f:
            json.dump(data, f, indent=2)

    def save(self, user):
        data = self._load()
        data[user["username"]] = user
        self._dump(data)

    def find(self, username):
        return self._load().get(username)


# --- Service depends on ABSTRACTION, not concrete class ---
class UserService:
    def __init__(self, repo: UserRepository):   # ← DIP: inject at runtime
        self._repo = repo

    def register(self, user: dict):
        self._repo.save(user)

    def get_user(self, username: str):
        return self._repo.find(username)


# Usage — swap the repo without changing UserService
repo = InMemoryUserRepository()
service = UserService(repo)
service.register({"username": "alice", "email": "a@b.com"})
print(service.get_user("alice"))
# {'username': 'alice', 'email': 'a@b.com'}

# Swap to JSON — UserService code is untouched
json_repo = JSONUserRepository()
service2 = UserService(json_repo)
service2.register({"username": "bob", "email": "b@b.com"})
print(service2.get_user("bob"))
# {'username': 'bob', 'email': 'b@b.com'}

{'username': 'alice', 'email': 'a@b.com'}
{'username': 'bob', 'email': 'b@b.com'}


### Q9. Race Condition — Shared Counter Fix 

Topics: Race Condition, Thread Safety 

Problem Statement: 

Create a shared counter starting at 0. Spawn 10 threads, each incrementing the counter 1000 times. First demonstrate the race condition (incorrect final value), then fix it using threading.Lock. 

Input: 

# No explicit input — internal simulation 

Output: 

Without lock: 8743 (varies, often incorrect) 

With lock:    10000 (always correct) 

Constraints: 

Run the unfixed version multiple times to show inconsistency 

Use threading.Lock to fix 

Do NOT use global keyword; pass lock and counter via a mutable container or class 

 

In [42]:
import threading
def increment_without_lock(container):
    for _ in range(1000):
        container[0]+=1
def increment_with_lock(container, lock):
    for _ in range(1000):
        with lock:             # only one thread at a time
            container[0] += 1
# --- Without lock (race condition) ---
counter = [0]                  # mutable container avoids 'global'
threads = [threading.Thread(target=increment_without_lock, args=(counter,))
           for _ in range(10)]
for t in threads: t.start()
for t in threads: t.join()
print(f"Without lock: {counter[0]}")   # e.g. 8743 — unpredictable!

# --- With lock (thread-safe) ---
counter = [0]
lock = threading.Lock()
threads = [threading.Thread(target=increment_with_lock, args=(counter, lock))
           for _ in range(10)]
for t in threads: t.start()
for t in threads: t.join()
print(f"With lock:    {counter[0]}")   # always 10000


Without lock: 10000
With lock:    10000


### Q10. Multiprocessing — CPU-bound Speedup 

Topics: Multiprocessing, CPU-bound 

Problem Statement: 

Write a function compute_squares(n) that computes the sum of squares from 1 to n. Run it for 4 large values sequentially and then using multiprocessing.Pool. Compare execution times. 

Input: 

values = [10_000_000, 20_000_000, 15_000_000, 25_000_000] 

Output: 

Sequential time: ~X.Xs 

Multiprocessing time: ~Y.Ys (faster) 

Constraints: 

Use multiprocessing.Pool.map() 

Measure wall-clock time with time.time() 

Print individual results for verification 

In [ ]:
import time
import multiprocessing
values=[10_000_000, 20_000_000, 15_000_000, 25_000_000]
def compute_squares(n):
    return sum(i * i for i in range(1, n + 1))

# --- Sequential ---
start = time.time()
results = [compute_squares(n) for n in values]
print(f"Results: {results}")
print(f"Sequential time: {time.time() - start:.2f}s")

# --- Multiprocessing ---
start = time.time()
with multiprocessing.Pool() as pool:
    results = pool.map(compute_squares, values)
print(f"Results: {results}")
print(f"Multiprocessing time: {time.time() - start:.2f}s")   # faster!

Results: [333333383333335000000, 2666666866666670000000, 1125000112500002500000, 5208333645833337500000]
Sequential time: 15.89s


### Q11. Async IO — Concurrent API Simulation 

Topics: Async, Event Loop, Coroutines 

Problem Statement: 

Convert synchronous API-call simulation into async using asyncio. Create async fetch(url, delay) that uses await asyncio.sleep(delay). Use asyncio.gather() to run 4 calls concurrently. Compare total time vs sync version. 

Input: 

urls = [("api/users", 2), ("api/orders", 3), ("api/products", 1), ("api/reviews", 2)] 

Output: 

Sync time:  ~8.0s 

Async time: ~3.0s 

Constraints: 

Use async def and await 

Use asyncio.gather() to run tasks concurrently 

Print each task's start and completion timestamp 

In [5]:
import asyncio, time

urls = [
    ("api/users", 2),
    ("api/orders", 3),
    ("api/products", 1),
    ("api/reviews", 2)
]

async def fetch(url, delay):
    ts = time.strftime("%H:%M:%S")
    print(f"[{ts}] START {url}")
    await asyncio.sleep(delay)
    ts = time.strftime("%H:%M:%S")
    print(f"[{ts}] DONE {url}")
    return url


def sync_version():
    start = time.time()
    for url, delay in urls:
        time.sleep(delay)
    print(f"Sync time: {time.time()-start:.1f}s")


async def main():
    start = time.time()
    await asyncio.gather(*[fetch(url, d) for url, d in urls])
    print(f"Async time: {time.time()-start:.1f}s")


sync_version()
await main()

Sync time: 8.0s
[23:42:39] START api/users
[23:42:39] START api/orders
[23:42:39] START api/products
[23:42:39] START api/reviews
[23:42:40] DONE api/products
[23:42:41] DONE api/users
[23:42:41] DONE api/reviews
[23:42:42] DONE api/orders
Async time: 3.0s


### Q12. Pydantic — User Schema with Nested Validation 

Topics: Pydantic, Nested Models, Validation 

Problem Statement: 

Create Pydantic models: Address (street, city, zip_code), UserCreate (username, email, password, age, address). Add validations: email must contain '@', password min 8 chars, age 18-120, zip_code must be exactly 6 digits. Create UserResponse that excludes password. 

Input: 

data = {"username": "alice", "email": "alice@mail.com", 

  "password": "securepass", "age": 25, 

  "address": {"street": "MG Road", "city": "Bangalore", "zip_code": "560001"}} 

user = UserCreate(**data) 

print(UserResponse(**user.model_dump())) 

Output: 

username='alice' email='alice@mail.com' age=25 address=Address(...) 

Constraints: 

Use field_validator or Field() constraints 

Invalid data must raise ValidationError with clear messages 

UserResponse must NOT include password field 

Use model_dump() for serialization 

In [8]:
from pydantic import BaseModel, field_validator, Field
from typing import Optional
class Address(BaseModel):
    street : str
    city:str
    zip_code:str

    @field_validator("zip_code")
    @classmethod
    def zip_must_be_6_digits(cls,v):
        if not (len(v)==6 and v.isdigit()):
            raise ValueError("zip_code must be exactly 6 digits")
        return v
class UserCreate(BaseModel):
    username :str
    email:str
    password :str =Field(min_length=8)
    age:int=Field(ge=18, le=120)
    address: Address

    @field_validator("email")
    @classmethod
    def email_must_have_at(cls,v):
        if '@' not in v:
            raise ValueError("email must contain '@")
        return v
class UserResponse(BaseModel):
    """Excludes password - safe to return to client"""
    username:str
    email: str
    age: int
    address:Address

data={
    "username":"sushma","email":"sushma@gmail.com",
    "password":"Securepass","age":25,
    "address":{"street":"MG road","city":"bangalore","zip_code":"560001"}
}

user =UserCreate(**data)
print(UserResponse(**user.model_dump()))

username='sushma' email='sushma@gmail.com' age=25 address=Address(street='MG road', city='bangalore', zip_code='560001')


### Q13. FastAPI — Health Check and CRUD Endpoints 

Topics: FastAPI, Pydantic, REST, HTTP Methods 

Problem Statement: 

Create a FastAPI application with the following endpoints for a Task resource (store tasks in a Python list in memory): 

GET /health — returns {"status": "healthy"} 

POST /tasks — create a new task 

GET /tasks — list all tasks with optional ?status= filter 

GET /tasks/{task_id} — get task by ID 

PUT /tasks/{task_id} — update a task 

DELETE /tasks/{task_id} — delete a task 

Input: 

# POST /tasks 

{"title": "Write report", "description": "Q2 summary", "status": "pending"} 

 

# GET /tasks?status=pending 

Output: 

# POST response (201) 

{"id": 1, "title": "Write report", "description": "Q2 summary", "status": "pending"} 

 

# GET response (200) 

[{"id": 1, "title": "Write report", ...}] 

Constraints: 

Use Pydantic models: TaskCreate, TaskUpdate, TaskResponse 

Task ID must be auto-generated (incrementing integer) 

Status must be one of: pending, in_progress, completed 

Return 404 if task not found 

Return 201 on successful creation 

Test all endpoints via Postman 

In [9]:
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel
from typing import Optional, List
from enum import Enum

app = FastAPI()

class StatusEnum(str, Enum):
    pending = "pending"
    in_progress = "in_progress"
    completed = "completed"

class TaskCreate(BaseModel):
    title: str
    description: str
    status: StatusEnum = StatusEnum.pending

class TaskUpdate(BaseModel):
    title: Optional[str] = None
    description: Optional[str] = None
    status: Optional[StatusEnum] = None

class TaskResponse(BaseModel):
    id: int
    title: str
    description: str
    status: StatusEnum

# In-memory store
tasks: List[dict] = []
_id_counter = 0

def next_id():
    global _id_counter
    _id_counter += 1
    return _id_counter

def get_task_or_404(task_id: int):
    task = next((t for t in tasks if t["id"] == task_id), None)
    if not task:
        raise HTTPException(status_code=404, detail=f"Task {task_id} not found")
    return task

@app.get("/health")
def health():
    return {"status": "healthy"}

@app.post("/tasks", response_model=TaskResponse, status_code=201)
def create_task(task: TaskCreate):
    new_task = {"id": next_id(), **task.model_dump()}
    tasks.append(new_task)
    return new_task

@app.get("/tasks", response_model=List[TaskResponse])
def list_tasks(status: Optional[StatusEnum] = None):
    if status:
        return [t for t in tasks if t["status"] == status]
    return tasks

@app.get("/tasks/{task_id}", response_model=TaskResponse)
def get_task(task_id: int):
    return get_task_or_404(task_id)

@app.put("/tasks/{task_id}", response_model=TaskResponse)
def update_task(task_id: int, update: TaskUpdate):
    task = get_task_or_404(task_id)
    for field, value in update.model_dump(exclude_none=True).items():
        task[field] = value
    return task

@app.delete("/tasks/{task_id}", status_code=204)
def delete_task(task_id: int):
    task = get_task_or_404(task_id)
    tasks.remove(task)

### Q14. Custom Exception Handler in FastAPI 

Topics: FastAPI, Error Handling 

Problem Statement: 

Add a custom exception class TaskNotFoundError and register a global exception handler in FastAPI that returns a structured JSON error response with proper status code. 

Input: 

# GET /tasks/999 (non-existent) 

Output: 

# Response (404) 

{"error": "TaskNotFoundError", "message": "Task with id 999 not found", "status_code": 404} 

Constraints: 

Create a custom exception class inheriting from Exception 

Use @app.exception_handler() decorator 

Response must always have error, message, and status_code fields 

Apply this handler to all relevant endpoints 

In [11]:
from fastapi import Request
from fastapi.responses import JSONResponse

class TaskNotFoundError(Exception):
    def __init__(self, task_id: int):
        self.task_id = task_id
        super().__init__(f"Task with id {task_id} not found")

@app.exception_handler(TaskNotFoundError)
async def task_not_found_handler(request: Request, exc: TaskNotFoundError):
    return JSONResponse(
        status_code=404,
        content={
            "error": "TaskNotFoundError",
            "message": str(exc),
            "status_code": 404
        }
    )

# Update get_task_or_404 to raise this custom error:
def get_task_or_404(task_id: int):
    task = next((t for t in tasks if t["id"] == task_id), None)
    if not task:
        raise TaskNotFoundError(task_id)   # ← raises custom, not HTTPException
    return task
# GET /tasks/999 → {"error": "TaskNotFoundError", "message": "Task with id 999 not found", "status_code": 404}

### Q15. Request Logging Middleware 

Topics: FastAPI, Middleware, Logging 

Problem Statement: 

Add a middleware to the FastAPI app that logs every incoming request with: timestamp, HTTP method, path, and response status code. Write logs to api_logs.txt. 

Input: 

# Any request: GET /tasks 

Output: 

# Log entry in api_logs.txt: 

2026-03-20 14:30:00 | GET /tasks | Status: 200 | Time: 12ms 

Constraints: 

Use @app.middleware("http") 

Calculate response time in milliseconds 

Log to file, not just console 

Include timestamp, method, path, status code, and duration 

In [12]:
import time
from datetime import datetime
from fastapi import Request

@app.middleware("http")
async def log_requests(request: Request, call_next):
    start = time.time()
    response = await call_next(request)
    duration_ms = int((time.time() - start) * 1000)

    log_line = (
        f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | "
        f"{request.method} {request.url.path} | "
        f"Status: {response.status_code} | "
        f"Time: {duration_ms}ms\n"
    )
    with open("api_logs.txt", "a") as f:
        f.write(log_line)
    print(log_line, end="")
    return response
# Log entry: 2026-03-22 10:15:00 | GET /tasks | Status: 200 | Time: 12ms

### Q16. Environment Variables and Config 

Topics: Config, Environment Variables, Pydantic 

Problem Statement: 

Create a Settings class using Pydantic's BaseSettings that loads APP_NAME, DEBUG, JSON_DB_PATH, and LOG_LEVEL from a .env file. Use these settings in your FastAPI app startup. 

Input: 

# .env file content: 

APP_NAME=TaskAPI 

DEBUG=true 

JSON_DB_PATH=./data/tasks.json 

LOG_LEVEL=INFO 

Output: 

# On app startup: 

App: TaskAPI | Debug: True | DB: ./data/tasks.json 

Constraints: 

Use pydantic-settings package 

Do NOT hardcode any config values in the source code 

Load settings using model_config = SettingsConfigDict(env_file=".env") 

Settings must be a singleton used across the app 

 